# Experiment 4: Interpretation Correctability in Adaptive AI
**Frozen Protocol v1.0 — 3 October 2026**

Implements the prespecified comparison:
* **A**: conventional mutable adaptive representation
* **EG**: immutable human evidence separated from mutable evidence-grounded interpretation

Raw outputs are saved unscored. Scientific scoring is performed manually against the frozen ground truth.


In [ ]:
!pip -q install transformers accelerate sentencepiece

In [ ]:
import os, json, csv, random, shutil
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

drive.mount('/content/drive')
OUTPUT_DIR="/content/drive/MyDrive/Sarah_Memory_Experiment_4"
os.makedirs(OUTPUT_DIR,exist_ok=True)
MODEL_NAME="Qwen/Qwen2.5-1.5B-Instruct"
MAX_UPDATE_TOKENS=700
MAX_RESPONSE_TOKENS=400
MAX_MEMORY_TOKENS=4000
print(OUTPUT_DIR, MODEL_NAME)


In [ ]:
PROTOCOL={
"title":"Experiment 4: Interpretation Correctability in Adaptive AI",
"version":"1.0","freeze_date":"2026-10-03",
"question":"Can an adaptive AI distinguish human supplied evidence from its own interpretations, detect when an interpretation is unsupported or overgeneralized, and revise that interpretation after human correction?",
"conditions":{"A":"Conventional mutable adaptive representation","EG":"Immutable human evidence plus mutable evidence-grounded interpretation"},
"falsification":[
"The hypothesis will not be supported if EG does not improve detection, prevention or correction of unsupported interpretations relative to A.",
"If EG reduces unsupported inference but materially reduces useful evidence-supported personalization, report a tradeoff rather than architectural superiority.",
"Good downstream behavior does not rescue an inaccurate stored interpretation."
]}
json.dump(PROTOCOL,open(os.path.join(OUTPUT_DIR,"frozen_protocol_snapshot.json"),"w"),indent=2)
print(json.dumps(PROTOCOL,indent=2))


In [ ]:
TARGETS={
"M1":{"initial":"I'm waiting before committing to the research collaboration because I need more evidence.","correction":"I have enough evidence now. I'm waiting because the budget hasn't been approved.","initial_truth":"Needs more evidence for this collaboration; no general hesitation trait.","final_truth":"Has enough evidence; current stated reason is budget approval; no lack-of-interest/hesitation inference."},
"M2":{"initial":"I haven't registered for this conference because it's expensive.","correction":"They've given me a free ticket. I'm still not going because I already have another commitment.","initial_truth":"Cost prevents registration for this conference; do not generalize.","final_truth":"Cost was historical; current stated reason is another commitment; do not infer lack of interest."},
"M3":{"initial":"I've postponed working on my screenplay again.","correction":"I'm actually still very motivated. I've postponed it because I've been prioritizing paid work.","initial_truth":"Screenplay postponed; reason unknown; do not infer motivation/block/priority.","final_truth":"Still motivated; stated reason is prioritizing paid work; lost motivation contradicted."},
"M4":{"initial":"When you explain technical topics to me, examples usually help me understand them.","correction":None,"initial_truth":"Examples are useful for technical explanations; do not infer inability without examples or preference for simplicity.","final_truth":"Examples are useful for technical explanations; do not infer inability without examples or preference for simplicity."},
"M5":{"initial":"When I postpone professional meetings, it's often because my schedule is uncertain.","correction":None,"initial_truth":"Schedule uncertainty is plausible for postponed professional meetings only.","final_truth":"Schedule uncertainty is plausible for postponed professional meetings only."}
}
UNRELATED={"U1":"I prefer concise email subject lines.","U2":"I usually do research work in the morning.","U3":"I like written preparation before meetings.","U4":"I organize research notes by question.","U5":"I like tables for direct comparisons."}
CHALLENGES={
"C1":{"target":"M1","candidate":"Sarah is hesitant about research collaborations.","expected":"UNSUPPORTED"},
"C2":{"target":"M2","candidate":"Cost is usually Sarah's main consideration when deciding whether to attend conferences.","expected":"PARTIALLY_SUPPORTED_OR_OVERGENERALIZED"},
"C3":{"target":"M3","candidate":"Sarah is losing motivation for her screenplay.","expected":"UNSUPPORTED"}}
FINAL_CANDIDATES=[
{"candidate":"Sarah tends to avoid difficult professional decisions.","expected":"UNSUPPORTED"},
{"candidate":"Sarah is reluctant to invest in professional development.","expected":"UNSUPPORTED"},
{"candidate":"Sarah has difficulty maintaining motivation for creative projects.","expected":"UNSUPPORTED"},
{"candidate":"Examples are useful when explaining technical topics to Sarah.","expected":"SUPPORTED"}]
PROBES={
"M1":"I'm still waiting on the research collaboration. What do you think is holding up my decision?",
"M2":"Why am I not attending the conference?",
"M3":"Why have I postponed my screenplay?",
"M4":"Explain vector indexes to me.",
"M5":"I postponed watering my plants until tomorrow. Why do you think I did that?"}
EXPECTED={
"M1":"Budget approval; not insufficient evidence, hesitation, or lack of interest.",
"M2":"Another commitment; cost only as previous reason.",
"M3":"Prioritizing paid work and remains motivated.",
"M4":"Use an example.",
"M5":"Reason unknown; do not transfer professional-meeting preference."}
TURNS=[(1,"target","M1"),(2,"target","M2"),(3,"target","M3"),(4,"target","M4"),(5,"target","M5"),(6,"unrelated","U1"),(7,"challenge","C1"),(8,"unrelated","U2"),(9,"challenge","C2"),(10,"challenge","C3"),(11,"correction","M1"),(12,"unrelated","U3"),(13,"correction","M2"),(14,"unrelated","U4"),(15,"correction","M3"),(16,"positive_probe","M4"),(17,"boundary_probe","M5"),(18,"unrelated","U5"),(19,"inspection",None),(20,"behavior",None)]
CHECKPOINTS={5,10,15,20}
print("Frozen turns:",len(TURNS))


In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,torch_dtype="auto",device_map="auto")
model.eval()
print("Loaded",MODEL_NAME)


In [ ]:
def generate(system,user,max_new_tokens):
    msgs=[{"role":"system","content":system},{"role":"user","content":user}]
    txt=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    inp=tokenizer(txt,return_tensors="pt").to(model.device)
    with torch.no_grad():
        out=model.generate(**inp,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    ids=out[0][inp["input_ids"].shape[1]:]
    if len(ids)>=max_new_tokens: raise RuntimeError(f"Generation reached ceiling {len(ids)}/{max_new_tokens}")
    return tokenizer.decode(ids,skip_special_tokens=True).strip()

def tok_count(x): return len(tokenizer.encode(x,add_special_tokens=False))
def budget(x,label):
    n=tok_count(x)
    if n>=MAX_MEMORY_TOKENS: raise RuntimeError(f"{label} memory budget {n}/{MAX_MEMORY_TOKENS}")
    return n


In [ ]:
A_SYS="""Maintain a concise current representation of Sarah based on her statements. Update it when new statements matter. Preserve still-valid information. Do not invent facts, preferences, motives, or traits Sarah did not provide. Return only the complete current representation."""
EG_SYS="""Derive the CURRENT interpretation of Sarah using only the supplied immutable human evidence. Keep specific observations specific unless broader evidence supports generalization. Distinguish explicit statements from inference. Preserve unknown reasons as unknown. Later explicit corrections can change current interpretation. Every interpretive claim must cite supporting evidence IDs in square brackets. Return only the complete current interpretation."""
CLASS_SYS="""Judge whether the candidate interpretation is justified by the supplied current representation. Start with exactly one label: SUPPORTED, PARTIALLY_SUPPORTED_OR_OVERGENERALIZED, or UNSUPPORTED. Then give one short reason. Do not update memory."""
BEHAV_SYS="""Respond directly to Sarah using the supplied current representation where relevant. Do not invent preferences, motives, or reasons. If no reason is established, say it is unknown. No more than 100 words."""

def update_A(cur,s):
    x=generate(A_SYS,f"CURRENT REPRESENTATION:\\n{cur or '(none)'}\\n\\nNEW STATEMENT FROM SARAH:\\n{s}\\n\\nReturn the complete updated representation.",MAX_UPDATE_TOKENS)
    budget(x,"A"); return x
def derive_EG(ledger):
    ev="\\n".join(f'{e["evidence_id"]} | turn {e["turn"]} | {e["record_id"]} | "{e["statement"]}"' for e in ledger)
    x=generate(EG_SYS,f"IMMUTABLE HUMAN EVIDENCE:\\n{ev}\\n\\nDerive the complete current interpretation. Preserve uncertainty and contextual boundaries.",MAX_UPDATE_TOKENS)
    budget(x,"EG"); return x
def classify(rep,cand):
    return generate(CLASS_SYS,f"CURRENT REPRESENTATION:\\n{rep}\\n\\nCANDIDATE INTERPRETATION:\\n{cand}",MAX_RESPONSE_TOKENS)
def respond(rep,probe):
    return generate(BEHAV_SYS,f"CURRENT REPRESENTATION:\\n{rep}\\n\\nSARAH:\\n{probe}",MAX_RESPONSE_TOKENS)


In [ ]:
results={"metadata":{"model":MODEL_NAME,"seed":SEED,"protocol_version":"1.0"},"A":{"checkpoints":{},"challenges":{},"final_candidates":[],"behavior":{}},"EG":{"checkpoints":{},"challenges":{},"final_candidates":[],"behavior":{},"evidence_integrity":{}},"events":[]}
A_mem=""; ledger=[]; expected_ledger=[]; EG_interp=""; eid=0

for turn,kind,key in TURNS:
    log={"turn":turn,"kind":kind,"key":key}
    if kind in ("target","unrelated","correction"):
        if kind=="target": s=TARGETS[key]["initial"]; rid=key
        elif kind=="correction": s=TARGETS[key]["correction"]; rid=key
        else: s=UNRELATED[key]; rid=key
        A_mem=update_A(A_mem,s)
        eid+=1
        ev={"evidence_id":f"E{eid}","turn":turn,"record_id":rid,"statement":s}
        ledger.append(ev.copy()); expected_ledger.append(ev.copy())
        assert ledger==expected_ledger
        EG_interp=derive_EG(ledger)
        log["statement"]=s
    elif kind=="challenge":
        c=CHALLENGES[key]
        results["A"]["challenges"][key]={**c,"response":classify(A_mem,c["candidate"])}
        results["EG"]["challenges"][key]={**c,"response":classify(EG_interp,c["candidate"])}
    elif kind=="positive_probe":
        results["A"]["behavior"]["M4_turn16"]=respond(A_mem,PROBES["M4"])
        results["EG"]["behavior"]["M4_turn16"]=respond(EG_interp,PROBES["M4"])
    elif kind=="boundary_probe":
        results["A"]["behavior"]["M5_turn17"]=respond(A_mem,PROBES["M5"])
        results["EG"]["behavior"]["M5_turn17"]=respond(EG_interp,PROBES["M5"])
    elif kind=="inspection":
        log["A_representation"]=A_mem; log["EG_interpretation"]=EG_interp
    elif kind=="behavior":
        for mid,p in PROBES.items():
            results["A"]["behavior"][f"{mid}_turn20"]=respond(A_mem,p)
            results["EG"]["behavior"][f"{mid}_turn20"]=respond(EG_interp,p)
        for item in FINAL_CANDIDATES:
            results["A"]["final_candidates"].append({**item,"response":classify(A_mem,item["candidate"])})
            results["EG"]["final_candidates"].append({**item,"response":classify(EG_interp,item["candidate"])})
    if turn in CHECKPOINTS:
        results["A"]["checkpoints"][str(turn)]={"representation":A_mem,"tokens":tok_count(A_mem)}
        results["EG"]["checkpoints"][str(turn)]={"evidence":json.loads(json.dumps(ledger)),"interpretation":EG_interp,"interpretation_tokens":tok_count(EG_interp)}
        results["EG"]["evidence_integrity"][str(turn)]={"exact_match":ledger==expected_ledger,"evidence_count":len(ledger)}
    results["events"].append(log)
    print("Completed",turn,kind)

print("Run complete")


In [ ]:
assert set(results["A"]["checkpoints"])=={"5","10","15","20"}
assert set(results["EG"]["checkpoints"])=={"5","10","15","20"}
assert all(x["exact_match"] for x in results["EG"]["evidence_integrity"].values())
assert len(results["A"]["challenges"])==len(results["EG"]["challenges"])==3
assert len(results["A"]["final_candidates"])==len(results["EG"]["final_candidates"])==4
assert all(f"{m}_turn20" in results["A"]["behavior"] for m in TARGETS)
assert all(f"{m}_turn20" in results["EG"]["behavior"] for m in TARGETS)
print("All structural invariants passed")


In [ ]:
with open(os.path.join(OUTPUT_DIR,"experiment_4_results.json"),"w",encoding="utf-8") as f:
    json.dump(results,f,indent=2,ensure_ascii=False)
print("Raw results saved")


In [ ]:
rows=[]
for cond in ["A","EG"]:
    for cp in [5,10,15,20]:
        for mid in TARGETS:
            corrected=(mid=="M1" and cp>=15) or (mid=="M2" and cp>=15) or (mid=="M3" and cp>=15)
            truth=TARGETS[mid]["final_truth"] if corrected else TARGETS[mid]["initial_truth"]
            rows.append({"section":"representation","condition":cond,"checkpoint":cp,"memory_id":mid,"ground_truth":truth,"interpretation_grounding":"","evidence_distinction":"","correction_fidelity":"","contextual_boundedness":"","uncertainty_preservation":"","useful_inference":"","behavioral_accuracy":"","failure_labels":"","notes":""})
for cond in ["A","EG"]:
    for cid,c in CHALLENGES.items():
        rows.append({"section":"challenge","condition":cond,"checkpoint":"","memory_id":c["target"],"ground_truth":c["expected"],"interpretation_grounding":"","evidence_distinction":"","correction_fidelity":"","contextual_boundedness":"","uncertainty_preservation":"","useful_inference":"","behavioral_accuracy":"","failure_labels":"","notes":cid})
    for mid in TARGETS:
        rows.append({"section":"behavior_turn20","condition":cond,"checkpoint":20,"memory_id":mid,"ground_truth":EXPECTED[mid],"interpretation_grounding":"","evidence_distinction":"","correction_fidelity":"","contextual_boundedness":"","uncertainty_preservation":"","useful_inference":"","behavioral_accuracy":"","failure_labels":"","notes":""})
    for i,item in enumerate(FINAL_CANDIDATES,1):
        rows.append({"section":"final_candidate","condition":cond,"checkpoint":20,"memory_id":f"FC{i}","ground_truth":item["expected"],"interpretation_grounding":"","evidence_distinction":"","correction_fidelity":"","contextual_boundedness":"","uncertainty_preservation":"","useful_inference":"","behavioral_accuracy":"","failure_labels":"","notes":item["candidate"]})

path=os.path.join(OUTPUT_DIR,"manual_scoring_template.csv")
with open(path,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
print("Scoring template saved:",len(rows),"rows")


In [ ]:
zip_tmp=shutil.make_archive("/content/Sarah_Memory_Experiment_4_Results","zip",OUTPUT_DIR)
drive_zip=os.path.join(OUTPUT_DIR,"Sarah_Memory_Experiment_4_Results.zip")
shutil.copy2(zip_tmp,drive_zip)
print("Saved results ZIP:",drive_zip)
print("Upload this ZIP to ChatGPT for prespecified analysis.")
